# CUDA 101: The CUDA Kernel

In [273]:
%%bash
nvidia-smi

Sun Oct 11 02:11:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.57                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:0C:00.0 Off |                  N/A |
|  0%   43C    P8             23W /  350W |    1433MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [274]:
%%bash
for file in *.c *.h *.cpp *.hpp *.cu *.cuh *.so *.py threads*; do
    rm -rf "$file"
done 

In [275]:
%%writefile parse_num.hpp
#pragma once
#include <charconv>
#include <cstdlib>
#include <string_view>
#include <system_error>
#include <tuple>

template <typename T>
std::tuple<T, int> parse_num(const std::string_view arg) {
  auto arg_end = arg.data() + arg.size();
  T val = 0;
  auto [read_end, read_err] = std::from_chars(arg.data(), arg_end, val);
  bool parse_err = read_err != std::errc{};
  bool partial_read = read_end != arg_end;
  if (parse_err || partial_read) {
    return {T{}, EXIT_FAILURE};
  }
  return {val, EXIT_SUCCESS};
}

Writing parse_num.hpp


In [276]:
%%writefile set_grid_shape.cuh
#pragma once
#include <cstdlib>
#include <cstdint>
#include <string_view>
#include <utility>
#include <cuda_runtime.h>
#include "parse_num.hpp"

inline int check_grid_shape(dim3& grid_shape, const cudaDeviceProp& prop) {
  if (grid_shape.x == 0 || grid_shape.y == 0 || grid_shape.z == 0) {
    return EXIT_FAILURE;
  }
  if (grid_shape.x > prop.maxGridSize[0] || grid_shape.y > prop.maxGridSize[1] || grid_shape.z > prop.maxGridSize[2]) {
    return EXIT_FAILURE;
  }
  return EXIT_SUCCESS;
}

Writing set_grid_shape.cuh


In [277]:
%%writefile set_block_shape.cuh
#pragma once
#include <cstdlib>
#include <cstdint>
#include <string_view>
#include <tuple>
#include <cuda_runtime.h>
#include "parse_num.hpp"

inline int check_block_shape(dim3& block_shape, const cudaDeviceProp& prop) {
  if (block_shape.x == 0 || block_shape.y == 0 || block_shape.z == 0) {
    return EXIT_FAILURE;
  }
  auto threads_per_block = block_shape.x * block_shape.y * block_shape.z;
  if (threads_per_block > prop.maxThreadsPerBlock) {
    return EXIT_FAILURE;
  }
  return EXIT_SUCCESS;
}

Writing set_block_shape.cuh


In [278]:
%%writefile get_prop.cuh
#pragma once
#include <cstdlib>
#include <cstdint>
#include <tuple>
#include <cuda_runtime.h>

inline std::tuple<cudaDeviceProp, int> get_prop(uint32_t device_id) {
  cudaDeviceProp prop;
  cudaError_t err = cudaGetDeviceProperties(&prop, device_id);
  if (err != cudaSuccess) {
    return {cudaDeviceProp{}, EXIT_FAILURE};
  }
  return {prop, EXIT_SUCCESS};
}

Writing get_prop.cuh


In [279]:
%%writefile threads_homogeneous.cuh
#pragma once
#include <cstdio>

__host__ __device__
void print(const char* msg) {
  printf("%s\n", msg);
}

Writing threads_homogeneous.cuh


In [280]:
%%writefile threads_host.cuh
#pragma once
#include <string>
#include "threads_homogeneous.cuh"

__host__
void welcome() {
  std::string msg = "This program will launch a GPU kernel and print its thread number";
  print(msg.c_str());
}

Writing threads_host.cuh


In [281]:
%%writefile threads_device.cuh
#pragma once
#include <cstdint>

__device__
uint32_t get_thread() {
  return blockIdx.x * blockDim.x + threadIdx.x;
}

Writing threads_device.cuh


In [282]:
%%writefile threads_kernel.cuh
#pragma once
#include <cstdio>
#include "threads_device.cuh"
#include "threads_homogeneous.cuh"

__global__
void kernel() {
  uint32_t thread = get_thread();
  printf("Hello from thread %u!\n", thread);
}

Writing threads_kernel.cuh


In [ ]:
%%writefile threads_run.cuh
#pragma once
#include <cstdint>
#include <cstdlib>
#include <format>
#include <iostream>
#include <string>
#include <cuda_runtime.h>
#include "get_prop.cuh"
#include "set_block_shape.cuh"
#include "set_grid_shape.cuh"
#include "threads_host.cuh"
#include "threads_kernel.cuh"

inline int run_threads(uint32_t grid_size, uint32_t block_size) {
  constexpr uint32_t device_id = 0;

  auto set_device_err = cudaSetDevice(device_id);
  if (set_device_err != cudaSuccess) {
    std::cerr << "Error setting device" << std::endl;
    return EXIT_FAILURE;
  }

  auto [prop, prop_err] = get_prop(device_id);
  if (prop_err != EXIT_SUCCESS) {
    std::cerr << "Error getting device properties" << std::endl;
    return EXIT_FAILURE;
  }

  auto grid_shape = dim3(grid_size, 1, 1);
  auto grid_shape_err = check_grid_shape(grid_shape, prop);
  if (grid_shape_err != EXIT_SUCCESS) {
    std::cerr << "Invalid grid shape" << std::endl;
    return EXIT_FAILURE;
  }

  auto block_shape = dim3(block_size, 1, 1);
  auto block_shape_err = check_block_shape(block_shape, prop);
  if (block_shape_err != EXIT_SUCCESS) {
    std::cerr << "Invalid block shape" << std::endl;
    return EXIT_FAILURE;
  }

  welcome();
  kernel<<<grid_shape, block_shape>>>();

  auto kernel_err = cudaGetLastError();
  if (kernel_err != cudaSuccess) {
    std::string msg = std::format("Kernel launch failed: {}", cudaGetErrorString(kernel_err));
    std::cerr << msg << std::endl;
    return EXIT_FAILURE;
  }

  auto sync_err = cudaDeviceSynchronize();
  if (sync_err != cudaSuccess) {
    std::string msg = std::format("cudaDeviceSynchronize failed: {}", cudaGetErrorString(sync_err));
    std::cerr << msg << std::endl;
    return EXIT_FAILURE;
  }

  return EXIT_SUCCESS;
}


In [ ]:
%%writefile threads.cu
#include <cstdint>
#include <cstdlib>
#include <format>
#include <iostream>
#include <span>
#include <string>
#include "parse_num.hpp"
#include "threads_run.cuh"

int main(int argc, char* argv[]) {
  std::span<char*> args(argv, argc);

  if (args.size() != 3) {
    std::string msg = std::format("Usage: {} <grid size> <block size>", args[0]);
    std::cerr << msg << std::endl;
    std::exit(EXIT_FAILURE);
  }

  auto [grid_size, grid_size_err] = parse_num<uint32_t>(args[1]);
  if (grid_size_err != EXIT_SUCCESS) {
    std::cerr << "Invalid grid size" << std::endl;
    std::exit(EXIT_FAILURE);
  }

  auto [block_size, block_size_err] = parse_num<uint32_t>(args[2]);
  if (block_size_err != EXIT_SUCCESS) {
    std::cerr << "Invalid block size" << std::endl;
    std::exit(EXIT_FAILURE);
  }

  return run_threads(grid_size, block_size);
}


In [284]:
%%bash
nvcc -std=c++20 -arch=native -O3 "threads.cu" -o threads

In [285]:
%%bash
./threads 2 3

This program will launch a GPU kernel and print its thread number
Hello from thread 3!
Hello from thread 4!
Hello from thread 5!
Hello from thread 0!
Hello from thread 1!
Hello from thread 2!


# CUDA 102: Extending Python with C

In [ ]:
%%writefile lib_cuda.cuh
#pragma once
#include <cstdint>
#include "parse_num.hpp"

struct pair {
  union {
    uint32_t u32;
    int32_t i32;
    float f32;
  };
  int ec;
};

extern "C" {
  pair parse_u32(const char* str);
  pair parse_i32(const char* str);
  pair parse_f32(const char* str);
  int launch_threads(uint32_t grid_size, uint32_t block_size);
}

In [ ]:
%%writefile lib_cuda.cu
#include <cstdint>
#include <string_view>
#include "lib_cuda.cuh"
#include "threads_run.cuh"

extern "C" {
  pair parse_u32(const char* str) {
    auto [val, ec] = parse_num<uint32_t>(std::string_view(str));
    return pair{.u32 = val, .ec = ec};
  }

  pair parse_i32(const char* str) {
    auto [val, ec] = parse_num<int32_t>(std::string_view(str));
    return pair{.i32 = val, .ec = ec};
  }

  pair parse_f32(const char* str) {
    auto [val, ec] = parse_num<float>(std::string_view(str));
    return pair{.f32 = val, .ec = ec};
  }

  int launch_threads(uint32_t grid_size, uint32_t block_size) {
    return run_threads(grid_size, block_size);
  }
}

In [288]:
%%bash
nvcc -std=c++20 -arch=native -O3 --compiler-options '-fPIC' -shared lib_cuda.cu -o lib_cuda.so

In [ ]:
%%writefile mod_cuda.cpp
#define PY_SSIZE_T_CLEAN
#include <Python.h>
#include <cstdint>
#include "lib_cuda.cuh"

static PyObject* py_parse_u32(PyObject* self, PyObject* args) {
    const char* str;
    if (!PyArg_ParseTuple(args, "s", &str)) {
        return nullptr;
    }
    pair res = parse_u32(str);
    if (res.ec != 0) {
        PyErr_SetString(PyExc_ValueError, "Failed to parse input as uint32_t");
        return nullptr;
    }
    return PyLong_FromUnsignedLong(res.u32);
}

static PyObject* py_parse_i32(PyObject* self, PyObject* args) {
    const char* str;
    if (!PyArg_ParseTuple(args, "s", &str)) {
        return nullptr;
    }
    pair res = parse_i32(str);
    if (res.ec != 0) {
        PyErr_SetString(PyExc_ValueError, "Failed to parse input as int32_t");
        return nullptr;
    }
    return PyLong_FromLong(res.i32);
}

static PyObject* py_parse_f32(PyObject* self, PyObject* args) {
    const char* str;
    if (!PyArg_ParseTuple(args, "s", &str)) {
        return nullptr;
    }
    pair res = parse_f32(str);
    if (res.ec != 0) {
        PyErr_SetString(PyExc_ValueError, "Failed to parse input as float");
        return nullptr;
    }
    return PyFloat_FromDouble(res.f32);
}

static PyObject* py_launch_threads(PyObject* self, PyObject* args) {
    unsigned int grid_size;
    unsigned int block_size;
    if (!PyArg_ParseTuple(args, "II", &grid_size, &block_size)) {
        return nullptr;
    }
    return PyLong_FromLong(launch_threads(grid_size, block_size));
}

static PyMethodDef LibCudaMethods[] = {
    {"parse_u32", py_parse_u32, METH_VARARGS, ""},
    {"parse_i32",  py_parse_i32,  METH_VARARGS, ""},
    {"parse_f32",  py_parse_f32,  METH_VARARGS, ""},
    {"launch_threads", py_launch_threads, METH_VARARGS, ""},
    {nullptr, nullptr, 0, nullptr}
};

static struct PyModuleDef libcudamodule = {
    PyModuleDef_HEAD_INIT,
    "mod_cuda",
    "",
    -1,
    LibCudaMethods
};

PyMODINIT_FUNC PyInit_mod_cuda(void) {
    return PyModule_Create(&libcudamodule);
}

In [290]:
%%bash
PYTHON_CFLAGS=$(python3-config --cflags)
PYTHON_LDFLAGS=$(python3-config --ldflags --embed)
g++ -std=c++20 -O3 -fPIC -shared \
    ${PYTHON_CFLAGS} \
    mod_cuda.cpp \
    ./lib_cuda.so \
    -o mod_cuda.so \
    ${PYTHON_LDFLAGS}

In [ ]:
%%writefile threads.py
import sys
import mod_cuda

if __name__ == '__main__':
    if len(sys.argv) != 3:
        print(f"Usage: {sys.argv[0]} <grid size> <block size>", file=sys.stderr)
        sys.exit(1)

    try:
        grid_size = mod_cuda.parse_u32(sys.argv[1])
    except ValueError:
        print("Invalid grid size", file=sys.stderr)
        sys.exit(1)

    try:
        block_size = mod_cuda.parse_u32(sys.argv[2])
    except ValueError:
        print("Invalid block size", file=sys.stderr)
        sys.exit(1)

    sys.exit(mod_cuda.launch_threads(grid_size, block_size))


In [ ]:
%%bash
python3 threads.py 2 3
